# code vectordb

# 25/10

In [1]:
pip install -U langchain langchain-community langchain-core chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.3/469.3 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 91.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 M

In [1]:
!pip install -q pyvi
!pip install langchain_community
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 60.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.1/467.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.66
    Uninstalling langchain-core-0.3.66:
      Successfully uninstalled langchain-core-0.3.66
  Attempting uninstall: langchain-text-splitters
    Found existing installation: lan

In [4]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
import time
import uuid
from transformers import AutoTokenizer
import json

# ===== Khởi tạo embedding model =====
embedding_function = HuggingFaceEmbeddings(
    model_name="dangvantuan/vietnamese-document-embedding",
    model_kwargs={"device": "cuda", "trust_remote_code": True},  # thêm trust_remote_code
    encode_kwargs={"normalize_embeddings": True}
)

# ===== Load tokenizer để tính số token =====
tokenizer = AutoTokenizer.from_pretrained("dangvantuan/vietnamese-document-embedding")

def token_len(text):
    return len(tokenizer.encode(text))


def clean_metadata(meta: dict):
    """Chuyển mọi list/dict/... thành str để tránh lỗi khi upsert vào Chroma."""
    clean = {}
    for k, v in meta.items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            clean[k] = v
        else:
            clean[k] = str(v)
    return clean

# ===== Đọc file JSON đã có =====
def load_chunks_from_json(file_input):
    """Đọc file JSON có dạng [{id, text, ...}]"""
    print(f"Đang đọc dữ liệu từ {file_input} ...")
    with open(file_input, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Đã tải {len(data)} records từ file.")
    return data

# ===== Insert vào Chroma =====
def insert_into_chroma(db, records, db_name="chunks_db", batch_size=500, persist_every=2000):
    total = len(records)
    if total == 0:
        print(f"{db_name}: không có record nào để lưu.")
        return

    print(f"Bắt đầu lưu {total} records vào {db_name} ...")
    t0 = time.time()
    saved = 0

    for start_idx in range(0, total, batch_size):
        batch = records[start_idx:start_idx + batch_size]

        docs = [
            Document(
                page_content=rec["text"],
                metadata=clean_metadata({k: v for k, v in rec.items() if k != "text"})
            )
            for rec in batch
            if "text" in rec and rec["text"]  # tránh record trống
        ]
        ids = [str(rec["id"]) for rec in batch]

        db.add_documents(documents=docs, ids=ids)
        saved += len(batch)

        pct = int(saved * 100 / total)
        elapsed = time.time() - t0
        print(f"Đã lưu {saved}/{total} ({pct}%) | {elapsed:.1f}s", flush=True)

        if saved % persist_every == 0:
            db.persist()
            print("Persist tạm thời xuống disk.", flush=True)

    db.persist()
    elapsed_total = time.time() - t0
    print(f"Hoàn tất lưu {db_name} | Tổng thời gian: {elapsed_total:.1f}s")

    try:
        count_now = db._collection.count()
        print(f"{db_name} hiện có {count_now} documents.")
    except Exception:
        pass


# ===== Khởi tạo Chroma =====
CHROMA_PATH = "/kaggle/working/vector_db_30_10"

chunk_db = Chroma(
    persist_directory=CHROMA_PATH,
    collection_name="chunks_his_dest",
    embedding_function=embedding_function
)


/tmp/ipykernel_36/3136051436.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(
2025-10-30 06:40:28.113458: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761806428.289076      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761806428.338493      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registe

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

... (11 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

/tmp/ipykernel_36/3136051436.py:92: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chunk_db = Chroma(


In [5]:
# ===== Main =====
if __name__ == "__main__":
    history_chunks = load_chunks_from_json("/kaggle/input/new-chunk-25-10/chunks_hist.json")
    destination_chunks = load_chunks_from_json("/kaggle/input/new-chunk-25-10/chunks_dest.json")

    # Lưu vào DB
    insert_into_chroma(chunk_db, history_chunks, "chunks_history", batch_size=50, persist_every=2000)
    insert_into_chroma(chunk_db, destination_chunks, "chunks_destination", batch_size=50, persist_every=2000)

    print("Hoàn tất: dữ liệu đã được split + embedding + lưu vào ChromaDB")


Đang đọc dữ liệu từ /kaggle/input/new-chunk-25-10/chunks_hist.json ...
Đã tải 7848 records từ file.
Đang đọc dữ liệu từ /kaggle/input/new-chunk-25-10/chunks_dest.json ...
Đã tải 1928 records từ file.
Bắt đầu lưu 7848 records vào chunks_history ...
Đã lưu 50/7848 (0%) | 2.9s
Đã lưu 100/7848 (1%) | 5.6s
Đã lưu 150/7848 (1%) | 8.0s
Đã lưu 200/7848 (2%) | 10.5s
Đã lưu 250/7848 (3%) | 13.0s
Đã lưu 300/7848 (3%) | 15.6s
Đã lưu 350/7848 (4%) | 18.3s
Đã lưu 400/7848 (5%) | 21.1s
Đã lưu 450/7848 (5%) | 23.7s
Đã lưu 500/7848 (6%) | 26.4s
Đã lưu 550/7848 (7%) | 29.3s
Đã lưu 600/7848 (7%) | 32.2s
Đã lưu 650/7848 (8%) | 34.8s
Đã lưu 700/7848 (8%) | 37.6s
Đã lưu 750/7848 (9%) | 40.4s
Đã lưu 800/7848 (10%) | 43.1s
Đã lưu 850/7848 (10%) | 45.8s
Đã lưu 900/7848 (11%) | 48.7s
Đã lưu 950/7848 (12%) | 51.8s
Đã lưu 1000/7848 (12%) | 54.9s
Đã lưu 1050/7848 (13%) | 57.7s
Đã lưu 1100/7848 (14%) | 60.3s
Đã lưu 1150/7848 (14%) | 63.2s
Đã lưu 1200/7848 (15%) | 66.5s
Đã lưu 1250/7848 (15%) | 69.5s
Đã lưu 1300/784

/tmp/ipykernel_36/3136051436.py:75: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


Đã lưu 2050/7848 (26%) | 119.5s
Đã lưu 2100/7848 (26%) | 122.5s
Đã lưu 2150/7848 (27%) | 125.6s
Đã lưu 2200/7848 (28%) | 128.6s
Đã lưu 2250/7848 (28%) | 131.7s
Đã lưu 2300/7848 (29%) | 134.6s
Đã lưu 2350/7848 (29%) | 137.6s
Đã lưu 2400/7848 (30%) | 140.4s
Đã lưu 2450/7848 (31%) | 143.5s
Đã lưu 2500/7848 (31%) | 146.9s
Đã lưu 2550/7848 (32%) | 149.7s
Đã lưu 2600/7848 (33%) | 152.7s
Đã lưu 2650/7848 (33%) | 155.8s
Đã lưu 2700/7848 (34%) | 159.0s
Đã lưu 2750/7848 (35%) | 162.0s
Đã lưu 2800/7848 (35%) | 165.3s
Đã lưu 2850/7848 (36%) | 168.9s
Đã lưu 2900/7848 (36%) | 172.4s
Đã lưu 2950/7848 (37%) | 175.6s
Đã lưu 3000/7848 (38%) | 179.2s
Đã lưu 3050/7848 (38%) | 182.3s
Đã lưu 3100/7848 (39%) | 185.5s
Đã lưu 3150/7848 (40%) | 188.7s
Đã lưu 3200/7848 (40%) | 191.6s
Đã lưu 3250/7848 (41%) | 194.8s
Đã lưu 3300/7848 (42%) | 197.9s
Đã lưu 3350/7848 (42%) | 201.1s
Đã lưu 3400/7848 (43%) | 204.5s
Đã lưu 3450/7848 (43%) | 207.9s
Đã lưu 3500/7848 (44%) | 211.4s
Đã lưu 3550/7848 (45%) | 214.8s
Đã lưu 3